# music_task_embed embedding summary

This directory contains the local downstream-task embedding pipeline for `music_dataset`, plus the extracted representations for all currently supported foundation models.

## Output layout

Embeddings are stored under:

```text
ismir2026/music_task_embed/embeddings/<dataset>/<task>/<model>/<representation>/<partition>.npz
```

Each `.npz` stores:

- `embeddings`
- `sample_id`
- `audio_relpath`
- `audio_path`
- `split`
- `fold`
- `target_json`
- `label_text`
- task / model metadata

Task manifests live under:

```text
ismir2026/music_task_embed/manifests/<dataset>/<task>/
```

## Extracted models at a glance

The full table is in [embedding_model_summary.csv](/home/hjkim/selective_auditory/ismir2026/music_task_embed/embedding_model_summary.csv).

| Model | HF checkpoint | Param count | Final dim | Saved `layer_*` count | Saved total reps |
|---|---|---:|---:|---:|---:|
| `clap` | `laion/clap-htsat-unfused` | ~153.6M | 512 | 5 | 6 |
| `clap_music` | `laion/larger_clap_music` | ~194.1M | 1024 | 5 | 6 |
| `clap_music_speech` | `laion/larger_clap_music_and_speech` | ~194.1M | 1024 | 5 | 6 |
| `mulan` | `OpenMuQ/MuQ-MuLan-large` | ~663.5M | 512 | 13 | 14 |
| `mert` | `m-a-p/MERT-v1-330M` | 330M | 1024 | 25 | 26 |
| `mert95` | `m-a-p/MERT-v1-95M` | 95M | 768 | 13 | 14 |
| `musicgen_small` | `facebook/musicgen-small` | 300M | 1024 | 25 | 26 |
| `musicgen_medium` | `facebook/musicgen-medium` | 1.5B | 1536 | 49 | 50 |
| `musicgen_large` | `facebook/musicgen-large` | 3.3B | 2048 | 49 | 50 |
| `musicgen_melody_large` | `facebook/musicgen-melody-large` | 3.3B | 2048 | 49 | 50 |

Parameter-count note:

- `MERT` and `MusicGen` counts above are the official checkpoint family scales.
- `CLAP` and `MuLAN` do not encode the count explicitly in the checkpoint name, so the values here are close approximations from the cached weight-file sizes.

## What was extracted for each model

### CLAP family

- `clap`
  - HF: <https://huggingface.co/laion/clap-htsat-unfused>
  - Final rep: projected `audio_embedding` (`512`-D)
  - Layer reps: pooled audio encoder hidden states `layer_00` to `layer_04`
  - Layer dims: `96 -> 768`
  - Input sample rate: `48 kHz`

- `clap_music`
  - HF: <https://huggingface.co/laion/larger_clap_music>
  - Final rep: encoder pooler output (`1024`-D), saved as `final`
  - Layer reps: pooled audio encoder hidden states `layer_00` to `layer_04`
  - Layer dims: `128 -> 1024`
  - Input sample rate: `48 kHz`

- `clap_music_speech`
  - HF: <https://huggingface.co/laion/larger_clap_music_and_speech>
  - Final rep: encoder pooler output (`1024`-D), saved as `final`
  - Layer reps: pooled audio encoder hidden states `layer_00` to `layer_04`
  - Layer dims: `128 -> 1024`
  - Input sample rate: `48 kHz`

Important difference:

- `clap` uses the projection-head output as the final representation.
- `clap_music` and `clap_music_speech` use the encoder output before the projection head.

### MuLAN

- HF: <https://huggingface.co/OpenMuQ/MuQ-MuLan-large>
- Final rep: pooled MuLAN audio latent (`512`-D)
- Layer reps: pooled backbone hidden states `layer_00` to `layer_12`
- Layer dims: all `1024`-D
- Input sample rate: `24 kHz`

### MERT

- `mert`
  - HF: <https://huggingface.co/m-a-p/MERT-v1-330M>
  - Final rep: pooled last hidden state (`1024`-D)
  - Layer reps: `layer_00` to `layer_24`
  - Layer dims: all `1024`-D
  - Input sample rate: `24 kHz`

- `mert95`
  - HF: <https://huggingface.co/m-a-p/MERT-v1-95M>
  - Final rep: pooled last hidden state (`768`-D)
  - Layer reps: `layer_00` to `layer_12`
  - Layer dims: all `768`-D
  - Input sample rate: `24 kHz`

Layer-count note:

- The saved MERT `layer_*` count includes the embedding-state plus each transformer block output.
- That is why the `330M` model gives `25` saved layers, while the `95M` model gives `13`.

### MusicGen decoder latents

- `musicgen_small`
  - HF: <https://huggingface.co/facebook/musicgen-small>
  - Official size: `300M`
  - Final rep: pooled last decoder hidden state (`1024`-D)
  - Layer reps: `layer_00` to `layer_24`
  - Layer dims: all `1024`-D
  - Input sample rate: `32 kHz`

- `musicgen_medium`
  - HF: <https://huggingface.co/facebook/musicgen-medium>
  - Official size: `1.5B`
  - Final rep: pooled last decoder hidden state (`1536`-D)
  - Layer reps: `layer_00` to `layer_48`
  - Layer dims: all `1536`-D
  - Input sample rate: `32 kHz`

- `musicgen_large`
  - HF: <https://huggingface.co/facebook/musicgen-large>
  - Official size: `3.3B`
  - Final rep: pooled last decoder hidden state (`2048`-D)
  - Layer reps: `layer_00` to `layer_48`
  - Layer dims: all `2048`-D
  - Input sample rate: `32 kHz`

- `musicgen_melody_large`
  - HF: <https://huggingface.co/facebook/musicgen-melody-large>
  - Official size: `3.3B`
  - Final rep: pooled last decoder hidden state (`2048`-D)
  - Layer reps: `layer_00` to `layer_48`
  - Layer dims: all `2048`-D
  - Input sample rate: `32 kHz`

Important extraction choice:

- We save MusicGen **decoder hidden states**, not EnCodec / codec latents.
- `final` is exactly the same tensor as the last saved decoder layer after sequence pooling.

## Why MusicGen looks like 25 / 49 / 49 layers instead of 24 / 48 / 48

The original decoder depths are still what you expected:

- `musicgen_small`: `24` decoder blocks
- `musicgen_medium`: `48` decoder blocks
- `musicgen_large`: `48` decoder blocks

The reason the saved `layer_*` count is one larger is that Hugging Face exposes:

```text
decoder_hidden_states = [
  input_embedding_state,
  block_01_output,
  block_02_output,
  ...,
  block_N_output,
]
```

So the saved layer counts become:

- `24 blocks -> 25 hidden states`
- `48 blocks -> 49 hidden states`

And because we also save a convenience copy named `final`, the total number of representation directories becomes:

- `25 hidden states + final = 26`
- `49 hidden states + final = 50`

This is not an off-by-one bug in the extraction. It is a naming / bookkeeping consequence of saving:

1. the decoder input state as `layer_00`
2. every block output as the following `layer_*`
3. the last pooled decoder state again as `final`

I also directly checked that:

- `musicgen_small/final == layer_24`
- `musicgen_medium/final == layer_48`
- `musicgen_large/final == layer_48`
- `musicgen_melody_large/final == layer_48`

with max absolute difference `0.0` in the saved arrays.

## music_dataset completion status

All supported models above have already been extracted for the six current downstream tasks:

- `emo/emotion_regression`
- `giantsteps/key_detection`
- `gtzan/genre_classification`
- `magnatagatune/music_tagging`
- `nsynthpitch5h/pitch_estimation`
- `vocalset/singing_technique`

The total saved `.npz` counts reflect the number of partitions times the number of saved representations:

- `clap`, `clap_music`, `clap_music_speech`: `120`
- `mulan`, `mert95`: `280`
- `mert`, `musicgen_small`: `520`
- `musicgen_medium`, `musicgen_large`, `musicgen_melody_large`: `1000`

## Resume safety

Extraction is resume-safe.

- If a run is interrupted, rerunning the same command will skip already completed partitions.
- Completion is checked against both the saved row count and the saved `sample_id` ordering.

## Main files

- [datasets.py](/home/hjkim/selective_auditory/ismir2026/music_task_embed/datasets.py): manifest construction
- [storage.py](/home/hjkim/selective_auditory/ismir2026/music_task_embed/storage.py): manifest I/O and partition save / resume checks
- [encoders.py](/home/hjkim/selective_auditory/ismir2026/music_task_embed/encoders.py): model loading and embedding extraction
- [extract_embeddings.py](/home/hjkim/selective_auditory/ismir2026/music_task_embed/extract_embeddings.py): single-manifest extraction entrypoint
- [run_extraction_job.py](/home/hjkim/selective_auditory/ismir2026/music_task_embed/run_extraction_job.py): multi-manifest batch runner

## Example command

```bash
conda run -n universal python -m ismir2026.music_task_embed.run_extraction_job \
  --output-root /home/hjkim/selective_auditory/ismir2026/music_task_embed/embeddings \
  --device cuda \
  --cache-dir /home/hjkim/.cache/huggingface \
  --local-files-only \
  --include-layers \
  --batch-size 1 \
  --model-name musicgen_large \
  --manifest-dir /home/hjkim/selective_auditory/ismir2026/music_task_embed/manifests/emo/emotion_regression \
  --manifest-dir /home/hjkim/selective_auditory/ismir2026/music_task_embed/manifests/giantsteps/key_detection \
  --manifest-dir /home/hjkim/selective_auditory/ismir2026/music_task_embed/manifests/gtzan/genre_classification \
  --manifest-dir /home/hjkim/selective_auditory/ismir2026/music_task_embed/manifests/magnatagatune/music_tagging \
  --manifest-dir /home/hjkim/selective_auditory/ismir2026/music_task_embed/manifests/nsynthpitch5h/pitch_estimation \
  --manifest-dir /home/hjkim/selective_auditory/ismir2026/music_task_embed/manifests/vocalset/singing_technique
```
